---
title: "ClimateKG Inventory Dashboard"
format:
  html:
    code-fold: true
execute:
  enabled: false
---

# ClimateKG Inventory Dashboard

Summary counts of key data items in the IPCC AR6 Climate Knowledge Graph.

**Data Sources:** Wikibase SPARQL endpoint (P1/Q3998 Authors, P1/Q6 Chapters, P10 DOI, P9 OpenAlex) + source CSV files  
**Instructions:** Run all cells with Wikibase running, then Quarto renders from stored outputs.


In [13]:
#| include: false
import pandas as pd
import plotly.graph_objects as go
from SPARQLWrapper import SPARQLWrapper, JSON
from IPython.display import display, HTML

print("âœ“ Libraries imported successfully")

âœ“ Libraries imported successfully


## Configuration

Update `WIKI_BASE_URL` and `SPARQL_ENDPOINT` for your environment.
CSV paths are relative to this notebook.

In [14]:
#| include: false
WIKI_BASE_URL   = "http://localhost:8080"
SPARQL_ENDPOINT = "http://localhost:9999/bigdata/namespace/wdq/sparql"

AUTHORS_CSV  = "../data-xml-dtd/authors-ar6.csv"
BIBLIO_CSV   = "../data-xml-dtd/bibliographic-ar6.csv"
GLOSSARY_CSV = "../data-xml-dtd/glossary-ar6.csv"
ACRONYMS_CSV = "../data-xml-dtd/acronyms-ar6.csv"

print(f"Wikibase: {WIKI_BASE_URL}")

Wikibase: http://localhost:8080


## Count Data

In [15]:
#| echo: false
def run_sparql(endpoint, query):
    """Run a SPARQL COUNT query; return integer count or None on error."""
    try:
        sp = SPARQLWrapper(endpoint)
        sp.setQuery(query)
        sp.setReturnFormat(JSON)
        r = sp.query().convert()
        return int(r["results"]["bindings"][0]["count"]["value"])
    except Exception:
        return None

PFX = (
    "PREFIX wdt: <" + WIKI_BASE_URL + "/prop/direct/> "
    "PREFIX wd:  <" + WIKI_BASE_URL + "/entity/>"
)

# Authors: COUNT DISTINCT P20 literal values (not entity URIs) â€” avoids duplicate Q-item import doubling
authors_count  = run_sparql(SPARQL_ENDPOINT, PFX + " SELECT (COUNT(DISTINCT ?uid) AS ?count) WHERE { ?item wdt:P20 ?uid }")
# Q6 = Chapter (instance type via P1)
chapters_count = run_sparql(SPARQL_ENDPOINT, PFX + " SELECT (COUNT(DISTINCT ?item) AS ?count) WHERE { ?item wdt:P1 wd:Q6 }")
# P10 = DOI
dois_count     = run_sparql(SPARQL_ENDPOINT, PFX + " SELECT (COUNT(DISTINCT ?item) AS ?count) WHERE { ?item wdt:P10 [] }")
# P9 = OpenAlex
openalex_count = run_sparql(SPARQL_ENDPOINT, PFX + " SELECT (COUNT(DISTINCT ?item) AS ?count) WHERE { ?item wdt:P9 [] }")

# â”€â”€ CSV fallbacks â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
df_a = pd.read_csv(AUTHORS_CSV)
if authors_count is None:
    authors_count = int(df_a["ClimateKG_Author_ID"].nunique())
if chapters_count is None:
    chapters_count = int(df_a["Chapter_QID"].nunique())

df_b = pd.read_csv(BIBLIO_CSV)
reports_count = int((df_b["Type"] == "Series").sum())

if dois_count is None:
    dois_count = int(df_b["DOI"].dropna().str.strip().ne("").sum())

if openalex_count is None:
    openalex_count = 0

df_g = pd.read_csv(GLOSSARY_CSV, header=0)
df_g = df_g[~df_g.iloc[:, 0].str.startswith("Instructions:", na=False)]
glossary_count = len(df_g)

df_acr = pd.read_csv(ACRONYMS_CSV, header=0)
acronyms_count = len(df_acr)

print(f"Authors:          {authors_count:,}")
print(f"Reports (Series): {reports_count:,}")
print(f"Chapters:         {chapters_count:,}")
print(f"DOIs:             {dois_count:,}")
print(f"OpenAlex IDs:     {openalex_count:,}")
print(f"Glossary Terms:   {glossary_count:,}")
print(f"Acronyms:         {acronyms_count:,}")


Authors:          932
Reports (Series): 7
Chapters:         88
DOIs:             95
OpenAlex IDs:     88
Glossary Terms:   920
Acronyms:         1,910


In [16]:
#| echo: false
metrics = [
    ("Authors",          authors_count),
    ("Reports (Series)", reports_count),
    ("Chapters",         chapters_count),
    ("DOIs",             dois_count),
    ("OpenAlex IDs",     openalex_count),
    ("Glossary Terms",   glossary_count),
    ("Acronyms",         acronyms_count),
]

fig = go.Figure()

# Row 1: first 4 metrics
for i, (label, value) in enumerate(metrics[:4]):
    fig.add_trace(go.Indicator(
        mode="number",
        value=float(value),
        title={"text": label, "font": {"size": 15, "color": "#555555"}},
        number={"font": {"size": 50, "color": "#2E86AB"}, "valueformat": ",d"},
        domain={"x": [i / 4, (i + 1) / 4], "y": [0.55, 1.0]},
    ))

# Row 2: last 3 metrics
for i, (label, value) in enumerate(metrics[4:]):
    fig.add_trace(go.Indicator(
        mode="number",
        value=float(value),
        title={"text": label, "font": {"size": 15, "color": "#555555"}},
        number={"font": {"size": 50, "color": "#2E86AB"}, "valueformat": ",d"},
        domain={"x": [i / 3, (i + 1) / 3], "y": [0.0, 0.45]},
    ))

fig.update_layout(
    title={"text": "Climate KG - Inventory Summary", "font": {"size": 18}, "x": 0.5},
    height=400,
    paper_bgcolor="#f8f9fa",
    margin={"t": 60, "b": 10, "l": 10, "r": 10},
)

display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

In [ ]:
#| echo: false
# -- Wikibase author Q-items table (all entries: original + duplicate batches) --
WIKI = WIKI_BASE_URL

dup_query = (
    "PREFIX wdt: <" + WIKI + "/prop/direct/> "
    "SELECT ?uid ?item WHERE { ?item wdt:P20 ?uid } ORDER BY ?uid ?item"
)

try:
    sp2 = SPARQLWrapper(SPARQL_ENDPOINT)
    sp2.setQuery(dup_query)
    sp2.setReturnFormat(JSON)
    bindings = sp2.query().convert()["results"]["bindings"]

    from collections import defaultdict
    uid_map = defaultdict(list)
    for b in bindings:
        uid_map[b["uid"]["value"]].append(b["item"]["value"].split("/")[-1])

    uid_col, qid_col, url_col, batch_col, action_col = [], [], [], [], []
    for uid in sorted(uid_map):
        items = sorted(uid_map[uid], key=lambda q: int(q[1:]))
        for i, qid in enumerate(items):
            uid_col.append(uid)
            qid_col.append(qid)
            url_col.append(f"{WIKI}/wiki/Item:{qid}")
            batch_col.append("Original" if i == 0 else "Duplicate")
            action_col.append("KEEP" if i == 0 else "DELETE")

    fig_t = go.Figure(data=[go.Table(
        columnwidth=[130, 80, 270, 90, 80],
        header=dict(
            values=["<b>Author ID</b>", "<b>QID</b>", "<b>Item URL</b>",
                    "<b>Batch</b>", "<b>Action</b>"],
            fill_color="#2E86AB",
            font=dict(color="white", size=12),
            align="left",
        ),
        cells=dict(
            values=[uid_col, qid_col, url_col, batch_col, action_col],
            fill_color=[["#ffffff" if i % 2 == 0 else "#f5f5f5" for i in range(len(uid_col))]],
            font=dict(size=11),
            align="left",
            height=22,
        ),
    )])
    fig_t.update_layout(
        title=dict(
            text=f"All Wikibase Author Q-items ({len(uid_col)} entries) -- Original: Q3999-Q4930 | Duplicate: Q4932-Q5863",
            font=dict(size=14), x=0.0),
        height=500,
        margin=dict(t=50, b=10, l=10, r=10),
    )
    display(HTML(fig_t.to_html(full_html=False, include_plotlyjs="cdn")))
    print(f"Table shows {len(uid_col)} Q-items. CSV: wikibase-duplicate-authors.csv")

except Exception as e:
    print(f"Could not load data from SPARQL: {e}")
    print("See wikibase-duplicate-authors.csv for the full list.")
